In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 10 — Ejercicio 1
# ---------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Cargar Mall Customers (igual que en el capítulo)
url = ("https://raw.githubusercontent.com/dsrscientist/dataset1/master/Mall_Customers.csv")
try:
    df = pd.read_csv(url)
except Exception:
    print("Usa el archivo descargado del capítulo")

X = df[["Annual Income (k$)", "Spending Score (1-100)"]].values
sc = StandardScaler()
X_sc = sc.fit_transform(X)

inercias, sil_scores = [], []
ks = range(2, 11)

for k in ks:
    km = KMeans(n_clusters=k, n_init="auto", random_state=42)
    etiq = km.fit_predict(X_sc)
    inercias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_sc, etiq))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(list(ks), inercias, marker="o")
axes[0].set_title("Método del codo")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inercia")

axes[1].plot(list(ks), sil_scores, marker="s", color="green")
axes[1].set_title("Silhouette score")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette")

plt.suptitle("Selección de k — Mall Customers")
plt.tight_layout()
plt.show()


In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 10 — Ejercicio 2
# ---------------------------------------------------------------

from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
import numpy as np

for eps in [0.2, 0.3, 0.4, 0.5, 0.6]:
    dbs = DBSCAN(eps=eps, min_samples=5)
    etiq = dbs.fit_predict(X_sc)
    n_clusters = len(set(etiq)) - (1 if -1 in etiq else 0)
    n_outliers  = (etiq == -1).sum()
    mask = etiq != -1
    sil = silhouette_score(X_sc[mask], etiq[mask]) if n_clusters > 1 else -1
    print(f"eps={eps} | clusters={n_clusters} |", f"outliers={n_outliers} | silhouette={sil:.3f}")

In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 10 — Ejercicio 3
# ---------------------------------------------------------------

from sklearn.datasets import load_digits
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score
import matplotlib.pyplot as plt
import numpy as np

X_dig, y_dig = load_digits(return_X_y=True)
scaler_dig = StandardScaler()
X_dig_sc = scaler_dig.fit_transform(X_dig)

km_dig = KMeans(n_clusters=10, n_init=20, random_state=42)
etiq_dig = km_dig.fit_predict(X_dig_sc)

ari = adjusted_rand_score(y_dig, etiq_dig)
print(f"Adjusted Rand Index: {ari:.4f}")

# Visualizar centroides como imágenes 8×8
centroides_orig = scaler_dig.inverse_transform(
    km_dig.cluster_centers_
)
fig, ejes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(ejes.flat):
    ax.imshow(centroides_orig[i].reshape(8, 8), cmap="gray_r")
    ax.axis("off")
plt.suptitle("Centroides K-Means — dígitos")
plt.tight_layout()
plt.show()
